In [2]:
import pandas as pd
### 所需要的所有文件：
# PLM的生命周期全表
# 物料基本信息_MARA，需要创建时间
# 近一年整机生产订单
# 近一年整机采购订单
# 所有整机的制造BOM

In [2]:
productline_list = ['油烟机产品线', '烹饪厨电产品线', '洗碗机产品线', '净热产品线','冰储产品线']

### 处理PLM导出的产品生命周期状态全表（保留13位物料号，国内，5大产品线）

In [3]:
df_plm = pd.read_excel(r"D:\000物料报表\物料运维报告\产品生命周期状态全表.xlsx")
len(df_plm)

e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


9136

In [4]:
df_plm['物料号'] = df_plm['物料号'].astype(str).str[:13]
df_plm = df_plm[df_plm['物料号'].str.len() == 13]
df_plm = df_plm[df_plm['国内/海外'] == '国内']
df_plm = df_plm[df_plm['产品线'].isin(productline_list)]
len(df_plm)
# df_plm.columns

6792

### 整机概览

In [5]:
# 创建初始DataFrame
df1 = pd.DataFrame()
df1['产品线'] = productline_list
# 定义状态列
status_columns = ['开发','样机','小批量','项目阶段小计','量产','退市预警',
                  '停止销售','在产阶段小计','停止生产','停止发货','停止服务',
                  '作废','淘汰阶段小计']
# 初始化状态列为0
df1[status_columns] = 0
# 遍历每条产品线
for index, row in df1.iterrows():
    product_line = row['产品线']
    # 遍历每个状态列
    for col in status_columns:
        # 统计符合条件的唯一物料号数量
        count = df_plm[(df_plm['产品线'] == product_line) & 
                      (df_plm['产品状态'] == col)]['物料号'].nunique()
        # 使用.loc更新DataFrame的值
        df1.loc[index, col] = count
# 计算各阶段小计
for index in df1.index:
    # 项目阶段小计 = 开发 + 样机 + 小批量
    df1.loc[index, '项目阶段小计'] = df1.loc[index, ['开发', '样机', '小批量']].sum()
    # 在产阶段小计 = 量产 + 退市预警 + 停止销售
    df1.loc[index, '在产阶段小计'] = df1.loc[index, ['量产', '退市预警', '停止销售']].sum()
    # 淘汰阶段小计 = 停止生产 + 停止发货 + 停止服务 + 作废
    df1.loc[index, '淘汰阶段小计'] = df1.loc[index, ['停止生产', '停止发货', '停止服务', '作废']].sum()
# 计算各列的合计值（排除'产品线'列）
total = df1.drop('产品线', axis=1).sum()
# 创建合计行
total_row = pd.DataFrame([total], index=['合计'])
total_row['产品线'] = '国内合计'  # 添加产品线标签
# 调整列顺序，保持与原DataFrame一致
total_row = total_row[df1.columns]
# 将合计行添加到原DataFrame末尾
df1_with_total = pd.concat([df1, total_row], ignore_index=True)
df1_with_total


,产品线,开发,样机,小批量,项目阶段小计,量产,退市预警,停止销售,在产阶段小计,停止生产,停止发货,停止服务,作废,淘汰阶段小计
0,油烟机产品线,3,12,38,53,262,13,96,371,21,438,97,72,628
1,烹饪厨电产品线,23,19,39,81,440,226,191,857,26,1713,0,10,1749
2,洗碗机产品线,4,6,18,28,105,19,70,194,2,82,0,15,99
3,净热产品线,0,4,6,10,113,33,24,170,15,799,0,15,829
4,冰储产品线,0,11,0,11,60,4,10,74,4,91,0,3,98
5,国内合计,30,52,101,183,980,295,391,1666,68,3123,97,115,3403


### 各产品线整机存活情况

#### 将创建日期匹配进PLM生命周期全表，并添加生命周期对应阶段列

In [ ]:
df_mara = pd.read_excel(r"D:\000物料报表\物料运维报告\物料基本信息_MARA.XLSX")
df_mara['物料编码'] = df_mara['物料编码'].astype(str).str[:13]
df_mara['制造方式'] = df_mara['制造方式'].astype(str).str[:2]
df_mara['跨工厂物料状态'] = df_mara['跨工厂物料状态'].astype(str).str[:1]
len(df_mara)


219533

In [35]:
df_plm_mara_left = pd.merge(df_plm, df_mara[['物料编码','创建日期']], left_on='物料号', right_on='物料编码', how='left')
len(df_plm_mara_left)


6792

In [36]:
df_plm_mara_left['生命周期对应阶段'] = df_plm_mara_left['产品状态'].apply(lambda x: '项目阶段' if x in ('开发','样机','小批量') else '在产阶段' if x in ('量产','退市预警','停止销售') else '淘汰阶段' if x in ('停止生产','停止发货','停止服务','作废') else '其他')
df_plm_mara_left['生命周期对应阶段'].value_counts()


生命周期对应阶段
淘汰阶段    3779
在产阶段    2824
项目阶段     189
Name: count, dtype: int64

#### 构建各产品线整机存活情况

In [37]:
df2 = pd.DataFrame()
df2['产品线'] = productline_list
for index,row in df2.iterrows():
    product_line = row['产品线']
    df2.loc[index,'2022年新增'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2022)]['物料号'].nunique()
    df2.loc[index,'2022年新增未淘汰'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2022) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()
    df2.loc[index,'2023年新增'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2023)]['物料号'].nunique()
    df2.loc[index,'2023年新增未淘汰'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2023) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()
    df2.loc[index,'2024年新增'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2024)]['物料号'].nunique()
    df2.loc[index,'2024年新增未淘汰'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2024) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()    
    df2.loc[index,'2025年新增'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2025)]['物料号'].nunique()
    df2.loc[index,'2025年新增未淘汰'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2025) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()
    df2.loc[index,'截至目前总型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line)]['物料号'].nunique()
    df2.loc[index,'截至目前未淘汰型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()  
df2
# 计算各列的合计值（排除'产品线'列）
total = df2.drop('产品线', axis=1).sum()
# 创建合计行
total_row = pd.DataFrame([total], index=['合计'])
total_row['产品线'] = '国内合计'  # 添加产品线标签
# 调整列顺序，保持与原DataFrame一致
total_row = total_row[df2.columns]
total_row
# 将合计行添加到原DataFrame末尾
df2_with_total = pd.concat([df2, total_row], ignore_index=True)
df2_with_total
# 计算2022年、2023年、2024年的存活率
df2_with_total['2022年存活率'] = df2_with_total['2022年新增未淘汰'] / df2_with_total['2022年新增']
df2_with_total['2023年存活率'] = df2_with_total['2023年新增未淘汰'] / df2_with_total['2023年新增']
df2_with_total['2024年存活率'] = df2_with_total['2024年新增未淘汰'] / df2_with_total['2024年新增']
df2_with_total['2025年存活率'] = df2_with_total['2025年新增未淘汰'] / df2_with_total['2025年新增']
df2_with_total



,产品线,2022年新增,2022年新增未淘汰,2023年新增,2023年新增未淘汰,2024年新增,2024年新增未淘汰,2025年新增,2025年新增未淘汰,截至目前总型号数,截至目前未淘汰型号数,2022年存活率,2023年存活率,2024年存活率,2025年存活率
0,油烟机产品线,75.0,65.0,46.0,43.0,71.0,71.0,98.0,98.0,1052.0,424.0,0.866667,0.934783,1.000000,1.0
1,烹饪厨电产品线,135.0,116.0,151.0,150.0,127.0,123.0,149.0,149.0,2687.0,938.0,0.859259,0.993377,0.968504,1.0
2,洗碗机产品线,26.0,18.0,59.0,52.0,56.0,48.0,39.0,39.0,321.0,222.0,0.692308,0.881356,0.857143,1.0
3,净热产品线,27.0,21.0,36.0,36.0,32.0,32.0,14.0,14.0,1009.0,180.0,0.777778,1.000000,1.000000,1.0
4,冰储产品线,9.0,6.0,10.0,9.0,26.0,26.0,12.0,12.0,183.0,85.0,0.666667,0.900000,1.000000,1.0
5,国内合计,272.0,226.0,302.0,290.0,312.0,300.0,312.0,312.0,5252.0,1849.0,0.830882,0.960265,0.961538,1.0


### 在产阶段，整机的生产,采购订单状况

In [38]:
df_maked = pd.read_excel(r"D:\000物料报表\物料运维报告\近一年整机生产订单.XLSX")
df_maked['物料编号'] = df_maked['物料编号'].astype(str).str[:13]
df_buyer = pd.read_excel(r"D:\000物料报表\物料运维报告\近一年整机采购订单.XLSX")
df_buyer['物料编码'] = df_buyer['物料编码'].astype(str).str[:13]
len(df_maked),len(df_buyer)


(43962, 29922)

#### 选出近一年生产过的物料编码，然后再新建列近一年是否有生产订单，来标记

In [39]:
maked_product = set(df_maked['物料编号'])
buyer_product = set(df_buyer['物料编码'])
df_plm_mara_left['近一年是否有生产订单'] = df_plm_mara_left['物料号'].apply(lambda x: '是' if x in maked_product else '否')
df_plm_mara_left['近一年是否有采购订单'] = df_plm_mara_left['物料号'].apply(lambda x: '是' if x in buyer_product else '否')
df_plm_mara_left['近一年是否有采购订单'].value_counts(),df_plm_mara_left['近一年是否有生产订单'].value_counts()


(近一年是否有采购订单
 否    5453
 是    1339
 Name: count, dtype: int64,
 近一年是否有生产订单
 否    4871
 是    1921
 Name: count, dtype: int64)

#### 构造整机在在产阶段的生产订单情况

In [40]:
df3 = pd.DataFrame()
status_columns = ['产品线','在产阶段的型号数','有生产订单的型号数','有采购订单的型号数','有采购&生产订单的型号数','有排单的型号数占比']
df3['产品线'] = productline_list
for index,row in df3.iterrows():
    product_line = row['产品线']
    df3.loc[index,'在产阶段的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '在产阶段')]['物料号'].nunique()
    df3.loc[index,'有生产订单的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '在产阶段') & (df_plm_mara_left['近一年是否有生产订单'] == '是')]['物料号'].nunique()
    df3.loc[index,'有采购订单的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '在产阶段') & (df_plm_mara_left['近一年是否有采购订单'] == '是')]['物料号'].nunique()
    df3.loc[index,'有采购生产订单的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '在产阶段') &  ((df_plm_mara_left['近一年是否有采购订单'] == '是') | (df_plm_mara_left['近一年是否有生产订单'] == '是'))]['物料号'].nunique()
df3

,产品线,在产阶段的型号数,有生产订单的型号数,有采购订单的型号数,有采购生产订单的型号数
0,油烟机产品线,371.0,224.0,82.0,241.0
1,烹饪厨电产品线,857.0,484.0,187.0,506.0
2,洗碗机产品线,194.0,157.0,71.0,168.0
3,净热产品线,170.0,122.0,79.0,140.0
4,冰储产品线,74.0,46.0,48.0,72.0


### 停止生产、发货阶段的整机统计

#### 读取产品的库存数据，并识别出哪些是有库存的整机，然后再新建列是否有库存，来标记

In [3]:
df_stock = pd.read_excel(r"D:\000物料报表\物料运维报告\整机仓库库存.XLSX")
df_stock['物料编码'] = df_stock['物料编码'].astype(str).str[:13]
df_stock['存储位置'] = df_stock['存储位置'].astype(str)
len(df_stock['物料编码'].unique())


2786

In [107]:
stored_product = set(df_stock[df_stock['存储位置'].str.startswith('3')]['物料编码'])
df_plm_mara_left['是否有库存'] = df_plm_mara_left['物料编码'].apply(lambda x: '是' if x in stored_product else '无')
df_plm_mara_left['是否有库存'].value_counts()


是否有库存
无    4742
是    2050
Name: count, dtype: int64

#### 构建淘汰阶段的库存情况统计

In [108]:
df4 = pd.DataFrame()
status_columns = ['产品线','淘汰阶段的型号数','有库存的型号数']
df4['产品线'] = productline_list
for index,row in df4.iterrows():
    product_line = row['产品线']
    df4.loc[index,'停止生产阶段的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['产品状态'] == '停止生产')]['物料号'].nunique()
    df4.loc[index,'停止生产阶段有库存的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['产品状态'] == '停止生产') & (df_plm_mara_left['是否有库存'] == '是')]['物料号'].nunique()
    df4.loc[index,'停止发货阶段的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['产品状态'] == '停止发货')]['物料号'].nunique()
    df4.loc[index,'停止发货阶段有库存的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['产品状态'] == '停止发货') & (df_plm_mara_left['是否有库存'] == '是')]['物料号'].nunique()
df4


,产品线,停止生产阶段的型号数,停止生产阶段有库存的型号数,停止发货阶段的型号数,停止发货阶段有库存的型号数
0,油烟机产品线,21.0,10.0,438.0,22.0
1,烹饪厨电产品线,26.0,7.0,1713.0,56.0
2,洗碗机产品线,2.0,0.0,82.0,12.0
3,净热产品线,15.0,8.0,799.0,31.0
4,冰储产品线,4.0,2.0,91.0,7.0


In [113]:
df_plm_mara_left[(df_plm_mara_left['产品状态'] == '停止发货') & (df_plm_mara_left['是否有库存'] == '是')][['物料编码','产品型号','产品状态']]


,物料编码,产品型号,产品状态
450,1001000900050,CXW-200-EM05G,停止发货
451,1001000900050,CXW-200-EM05G,停止发货
461,1001000900057,CXW-200-EM11T,停止发货
462,1001000900057,CXW-200-EM11T,停止发货
463,1001000900058,CXW-200-EM12T,停止发货
...,...,...,...
6773,1009000100010,ZW-C2S,停止发货
6779,1009000200010,ZW-Z1,停止发货
6780,1009000200010,ZW-Z1,停止发货
6782,1009000200011,ZW-Z2M7,停止发货


In [112]:
df_plm_mara_left.columns

Index(['产品线', '产品组', '标准型号', '是否标准型号', '简化型号', '国内/海外', '物料号', '关联项目', '产品所有者',
       '产品型号', '当前评审阶段', '产品状态', '产品状态开始时间', '下属渠道', '对应渠道状态', '开始销售时间',
       '退市预警时间', '停止销售时间', '停止发货时间', '剩余负卖生产数量', '物料编码', '创建日期', '生命周期对应阶段',
       '近一年是否有生产订单', '近一年是否有采购订单', '是否有库存'],
      dtype='object')

### 零件概览

In [62]:
# 10开头是整机、11：零部件、12：电子元器件、13：紧固密封件、14：板材、15：其他物料
for index,row in df_mara.iterrows():
    if row['物料编码'][:2] == '10':
        df_mara.loc[index,'物料类型'] = '整机'
    elif row['物料编码'][:2] == '11':
        df_mara.loc[index,'物料类型'] = '零部件'
    elif row['物料编码'][:2] == '12':
        df_mara.loc[index,'物料类型'] = '电子元器件'
    elif row['物料编码'][:2] == '13':
        df_mara.loc[index,'物料类型'] = '紧固密封件'
    elif row['物料编码'][:2] == '14':
        df_mara.loc[index,'物料类型'] = '板材'
    elif row['物料编码'][:2] == '15':
        df_mara.loc[index,'物料类型'] = '其他物料'


for index,row in df_mara.iterrows():
    if row['物料编码'][:4] in ['1101']:
        df_mara.loc[index,'零部件所用产品线'] = '油烟机产品线'
    if row['物料编码'][:4] in ['1102','1105','1106','1107','1109','1110','1111','1116']:
        df_mara.loc[index,'零部件所用产品线'] = '烹饪厨电产品线'
    if row['物料编码'][:4] in ['1108','1118','1124','1126']:
        df_mara.loc[index,'零部件所用产品线'] = '洗碗机产品线'
    if row['物料编码'][:4] in ['1104','1113','1114']:
        df_mara.loc[index,'零部件所用产品线'] = '净热产品线'
    if row['物料编码'][:4] in ['1103','1119']:
        df_mara.loc[index,'零部件所用产品线'] = '冰储产品线'
    if row['物料编码'][:2] in ['12','13','14','15']:
        df_mara.loc[index,'零部件所用产品线'] = 'all'
df_mara['物料类型'].value_counts(),df_mara['零部件所用产品线'].value_counts()

(物料类型
 零部件      131756
 其他物料      66164
 电子元器件      9161
 整机         7036
 板材         4257
 紧固密封件       488
 Z001        428
 Z003        220
 Z004         21
 Z002          2
 Name: count, dtype: int64,
 零部件所用产品线
 all        80070
 烹饪厨电产品线    47834
 油烟机产品线     36911
 净热产品线      18937
 洗碗机产品线     15623
 冰储产品线       7791
 Name: count, dtype: int64)

In [ ]:
from numpy import isin
df5 = pd.DataFrame()
df5['物料类型'] = ['零部件','零部件','零部件','零部件','零部件','电子元器件','紧固密封件','板材','其他物料']
df5['零部件所用产品线'] = ['油烟机产品线','烹饪厨电产品线','洗碗机产品线','净热产品线','冰储产品线','all','all','all','all']
for index,row in df5.iterrows():
    prodcut_type = row['物料类型']
    prodcut_type_line = row['零部件所用产品线']
    df5.loc[index,'自制数量'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & (df_mara['是否冻结'] != 'X')&(df_mara['制造方式']=='10')]['物料编码'].nunique()
    df5.loc[index,'外购&外协'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & (df_mara['是否冻结'] != 'X')&(df_mara['制造方式']!='10')]['物料编码'].nunique()
    df5.loc[index,'非冻结总计数量'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & (df_mara['是否冻结'] != 'X')]['物料编码'].nunique()
    df5.loc[index,'冻结数量'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & (df_mara['是否冻结'] == 'X')]['物料编码'].nunique()
    df5.loc[index,'冻结&采购冻结数量'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & ((df_mara['是否冻结'] == 'X')|(df_mara['跨工厂物料状态'].isin(['4','1','5'])))]['物料编码'].nunique()
df5

,物料类型,零部件所用产品线,自制数量,外购&外协,非冻结总计数量,冻结数量,冻结&采购冻结数量
0,零部件,油烟机产品线,6070.0,12080.0,18150.0,18761.0,20188.0
1,零部件,烹饪厨电产品线,6536.0,18329.0,24865.0,22969.0,24511.0
2,零部件,洗碗机产品线,1860.0,8225.0,10085.0,5538.0,5538.0
3,零部件,净热产品线,1375.0,7224.0,8599.0,10338.0,10397.0
4,零部件,冰储产品线,248.0,4463.0,4711.0,3080.0,3080.0
5,电子元器件,all,1303.0,5063.0,6366.0,2795.0,2795.0
6,紧固密封件,all,0.0,344.0,344.0,144.0,144.0
7,板材,all,0.0,3039.0,3039.0,1218.0,1218.0
8,其他物料,all,0.0,42178.0,42178.0,23986.0,23986.0


### 零部件分布

In [77]:
df6 = pd.DataFrame()
df6['物料类型'] = ['零部件','零部件','零部件','零部件','零部件']
df6['零部件所用产品线'] = ['油烟机产品线','烹饪厨电产品线','洗碗机产品线','净热产品线','冰储产品线']
for index,row in df6.iterrows():
    prodcut_type = row['物料类型']
    prodcut_type_line = row['零部件所用产品线']
    df6.loc[index,'2024年新增物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2024)]['物料编码'].nunique()
    df6.loc[index,'2024年变更物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2024)& (df_mara['物料编码'].str[-1].str.isalpha())]['物料编码'].nunique()
    df6.loc[index,'2025年第一季度新增物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2025) & (df_mara['创建日期'].dt.quarter == 1)]['物料编码'].nunique()
    df6.loc[index,'2025年第一季度变更物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2025) & (df_mara['创建日期'].dt.quarter == 1)& (df_mara['物料编码'].str[-1].str.isalpha())]['物料编码'].nunique()
    df6.loc[index,'2025年第二季度新增物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2025) & (df_mara['创建日期'].dt.quarter == 2)]['物料编码'].nunique()
    df6.loc[index,'2025年第二季度变更物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2025) & (df_mara['创建日期'].dt.quarter == 2)& (df_mara['物料编码'].str[-1].str.isalpha())]['物料编码'].nunique()
    df6.loc[index,'2025年第三季度新增物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2025) & (df_mara['创建日期'].dt.quarter == 3)]['物料编码'].nunique()
    df6.loc[index,'2025年第三季度变更物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2025) & (df_mara['创建日期'].dt.quarter == 3)& (df_mara['物料编码'].str[-1].str.isalpha())]['物料编码'].nunique()
    df6.loc[index,'2025年第四季度新增物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2025) & (df_mara['创建日期'].dt.quarter == 4)]['物料编码'].nunique()
    df6.loc[index,'2025年第四季度变更物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line)& (df_mara['创建日期'].dt.year == 2025) & (df_mara['创建日期'].dt.quarter == 4)& (df_mara['物料编码'].str[-1].str.isalpha())]['物料编码'].nunique()
df6


,物料类型,零部件所用产品线,2024年新增物料数,2024年变更物料数,2025年第一季度新增物料数,2025年第一季度变更物料数,2025年第二季度新增物料数,2025年第二季度变更物料数,2025年第三季度新增物料数,2025年第三季度变更物料数,2025年第四季度新增物料数,2025年第四季度变更物料数
0,零部件,油烟机产品线,3101.0,1566.0,1159.0,443.0,1137.0,335.0,1008.0,419.0,0.0,0.0
1,零部件,烹饪厨电产品线,3803.0,1439.0,1413.0,374.0,1052.0,499.0,1178.0,459.0,0.0,0.0
2,零部件,洗碗机产品线,2496.0,1229.0,579.0,210.0,887.0,330.0,556.0,323.0,0.0,0.0
3,零部件,净热产品线,1954.0,785.0,207.0,101.0,659.0,188.0,375.0,157.0,0.0,0.0
4,零部件,冰储产品线,1495.0,525.0,141.0,98.0,198.0,102.0,149.0,66.0,0.0,0.0


### 各产品线零件存货情况

In [ ]:
# 这里需要调整一下，应该换为有过生产或者采购订单的整机BOM
df_mbom = pd.read_excel(fr"D:\000物料报表\物料运维报告\有过生产订单的整机BOM.XLSX")
df_mbom['子项物料编码'] = df_mbom['子项物料编码'].astype(str).str[:13]


In [89]:
#先找出哪些零部件被使用过
parts_used = df_mbom['子项物料编码'].drop_duplicates()
parts_used = list(parts_used)
df7 = pd.DataFrame()
df7['物料类型'] = ['零部件','零部件','零部件','零部件','零部件']
df7['零部件所用产品线'] = ['油烟机产品线','烹饪厨电产品线','洗碗机产品线','净热产品线','冰储产品线']
for index, row in df7.iterrows():
    prodcut_type = row['物料类型']
    prodcut_type_line = row['零部件所用产品线']
    df7.loc[index, '2024年新增'] = df_mara[(df_mara['物料类型'] == prodcut_type)&(df_mara['零部件所用产品线'] == prodcut_type_line)&(df_mara['创建日期'].dt.year == 2024)]['物料编码'].nunique()
    df7.loc[index, '2024年新增物料使用的物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type)&(df_mara['零部件所用产品线'] == prodcut_type_line)&(df_mara['创建日期'].dt.year == 2024)&(df_mara['物料编码'].isin(parts_used))]['物料编码'].nunique()
    df7.loc[index, '2025年新增'] = df_mara[(df_mara['物料类型'] == prodcut_type)&(df_mara['零部件所用产品线'] == prodcut_type_line)&(df_mara['创建日期'].dt.year == 2025)]['物料编码'].nunique()
    df7.loc[index, '2025年新增物料使用的物料数'] = df_mara[(df_mara['物料类型'] == prodcut_type)&(df_mara['零部件所用产品线'] == prodcut_type_line)&(df_mara['创建日期'].dt.year == 2025)&(df_mara['物料编码'].isin(parts_used))]['物料编码'].nunique()

df7['2024年新增物料使用的物料数占比'] = df7['2024年新增物料使用的物料数'] / df7['2024年新增']
df7['2025年新增物料使用的物料数占比'] = df7['2025年新增物料使用的物料数'] / df7['2025年新增']
df7


,物料类型,零部件所用产品线,2024年新增,2024年新增物料使用的物料数,2025年新增,2025年新增物料使用的物料数,2024年新增物料使用的物料数占比,2025年新增物料使用的物料数占比
0,零部件,油烟机产品线,3101.0,1487.0,3304.0,1928.0,0.479523,0.583535
1,零部件,烹饪厨电产品线,3803.0,1968.0,3643.0,2239.0,0.517486,0.614603
2,零部件,洗碗机产品线,2496.0,1002.0,2022.0,1014.0,0.401442,0.501484
3,零部件,净热产品线,1954.0,847.0,1241.0,582.0,0.433470,0.468977
4,零部件,冰储产品线,1495.0,350.0,488.0,60.0,0.234114,0.122951


In [103]:
with pd.ExcelWriter(fr"D:\000物料报表\物料运维报告\物料运维报告.xlsx", engine='xlsxwriter') as writer:
    workbook = writer.book
    # 表头格式
    header_format = workbook.add_format({
        'font_name': '微软雅黑',
        'font_color': 'white',
        'bg_color': '#990000',
        'bold': True,
        'align': 'center',
        'valign': 'vcenter',
        'border': 1
    })
    
    # 数据格式
    data_format = workbook.add_format({
        'font_name': '微软雅黑',
        'border': 1
    })
    
    def write_and_style_fixed(df, sheet_name):
        """修复版本的write_and_style函数"""
        # 创建新的工作表
        worksheet = workbook.add_worksheet(sheet_name)
        
        # 手动写入表头并应用格式 - 从第一列开始
        for col_num, value in enumerate(df.columns.values):
            worksheet.write(0, col_num, value, header_format)
        
        # 写入数据并应用格式
        for row_num, row_data in enumerate(df.values, start=1):
            for col_num, cell_value in enumerate(row_data):
                worksheet.write(row_num, col_num, cell_value, data_format)
        
        # 精确计算列宽
        for col_num, col_name in enumerate(df.columns):
            # 计算列名长度和数据最大长度
            header_length = len(str(col_name))
            data_max_length = df[col_name].astype(str).str.len().max()
            max_length = max(header_length, data_max_length)
            
            # 设置合适的列宽（字符数 * 1.1 + 1.5作为缓冲）
            column_width = max_length * 1.1 + 1.5
            # 限制最小8，最大40
            column_width = min(max(column_width, 8), 40)
            worksheet.set_column(col_num, col_num, column_width)
        
        # 冻结首行
        worksheet.freeze_panes(1, 0)
        # 设置表头行高
        worksheet.set_row(0, 20)
    
    # 写入各个工作表
    write_and_style_fixed(df1_with_total, '整机概览')
    write_and_style_fixed(df2_with_total, '各产品线整机存活情况')
    write_and_style_fixed(df3, '在产型号生产采购订单情况')
    write_and_style_fixed(df4, '淘汰阶段型号采购订单情况')
    write_and_style_fixed(df5, '零件冻结&自制外购情况')
    write_and_style_fixed(df6, '零部件新增情况')
    write_and_style_fixed(df7, '新增零部件使用情况')

print("Excel文件已成功创建：物料运维报告.xlsx")


Excel文件已成功创建：物料运维报告.xlsx
